<a href="https://colab.research.google.com/github/Cyb3rVigil/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cyb3rVigil/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My chosen lane is **Lane 2: Refresh / Content Opportunity Scoring**.

The primary ML task is **ranking/scoring** because the final output is not only a yes-or-no prediction. The system must assign a score to each pseudonymized content page and order the pages so that an editor can review the most promising candidates first.

A binary classification model may later estimate the probability that a page will experience a meaningful later-window decline. That probability can become one component of the final opportunity score. However, the operational task remains ranking because the content team has limited capacity and needs an ordered top-50 review queue.

The output will contain a priority score, reason codes, a confidence level, and a suggested human action such as refresh, expand, protect, prune, or monitor.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Cyb3rVigil/flyrank-ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

task_frame = pd.DataFrame(
    {
        "item": [
            "Lane",
            "Primary ML task",
            "Scored item",
            "Operational output",
        ],
        "choice": [
            "Refresh / Content Opportunity Scoring",
            "Ranking / scoring",
            "One pseudonymized content page",
            "Top-50 human-review queue with reasons and actions",
        ],
    }
)

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns.")

task_frame

Loaded 30,000 rows and 44 columns.


,item,choice
0,Lane,Refresh / Content Opportunity Scoring
1,Primary ML task,Ranking / scoring
2,Scored item,One pseudonymized content page
3,Operational output,Top-50 human-review queue with reasons and act...


## 2. Target or proxy

The provisional target is a binary column named `future_decline_proxy`.

I define a decision boundary between two observed 30-day periods:

- The previous 30-day period provides the starting measurement.
- The later 30-day period provides the observed outcome.
- A page is eligible for the proxy only when it had at least 100 impressions in the previous period.
- For an eligible page, `future_decline_proxy = 1` when later impressions are at least 20% lower than previous-period impressions.
- Otherwise, `future_decline_proxy = 0`.
- Pages with fewer than 100 previous-period impressions receive a missing target rather than zero because there is not enough demand to separate meaningful decline from low-volume noise.

The observed outcome is the change in impressions. The 20% boundary and 100-impression minimum are provisional policy choices, not universal truths.

This proxy means “the page showed a meaningful later-window decline.” It does not mean that refreshing the page would cause recovery. In a later warehouse version, I would use features from a strictly earlier window and measure the target in a separate future window.

The outcome columns `impressions_last_30d`, `observed_impression_change_pct`, and `future_decline_proxy` must never be used as model features because that would leak the answer into the model. The existing `trend_direction` and `trend_pct` columns will also be excluded from features used to predict decline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
MIN_PRIOR_IMPRESSIONS = 100
DECLINE_THRESHOLD_PCT = -20.0

previous = df["impressions_prev_30d"]
later = df["impressions_last_30d"]

df["observed_impression_change_pct"] = np.where(
    previous.gt(0),
    100 * (later - previous) / previous,
    np.nan,
)

eligible = previous.ge(MIN_PRIOR_IMPRESSIONS)

df["future_decline_proxy"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="Int64",
)

df.loc[eligible, "future_decline_proxy"] = (
    df.loc[eligible, "observed_impression_change_pct"]
    .le(DECLINE_THRESHOLD_PCT)
    .astype(int)
    .to_numpy()
)

proxy_summary = pd.DataFrame(
    {
        "measure": [
            "All content pages",
            "Pages eligible for proxy label",
            "Eligible pages labelled decline=1",
        ],
        "count": [
            len(df),
            int(eligible.sum()),
            int(df.loc[eligible, "future_decline_proxy"].sum()),
        ],
    }
)

proxy_base_rate = float(
    df.loc[eligible, "future_decline_proxy"].mean()
)

print(proxy_summary.to_string(index=False))
print(
    f"\nPositive proxy rate among eligible pages: "
    f"{proxy_base_rate:.2%}"
)

df.loc[
    eligible,
    [
        "impressions_prev_30d",
        "impressions_last_30d",
        "observed_impression_change_pct",
        "future_decline_proxy",
    ],
].head(8)

                          measure  count
                All content pages  30000
   Pages eligible for proxy label  18010
Eligible pages labelled decline=1  11101

Positive proxy rate among eligible pages: 61.64%


,impressions_prev_30d,impressions_last_30d,observed_impression_change_pct,future_decline_proxy
0,987,578,-41.438703,1
1,5915,2501,-57.717667,1
2,6089,2382,-60.880276,1
3,4206,3626,-13.789824,0
4,6452,4211,-34.733416,1
5,1009,617,-38.850347,1
7,632,636,0.632911,0
8,13828,5696,-58.808215,1


## 3. Success metric

The primary success metric is **Precision@50** measured on held-out data.

`Precision@50` answers: among the 50 pages ranked highest by the system, what proportion actually meet the later-window decline proxy?

I chose 50 because it represents the assumed number of pages that a content team can inspect during one review cycle. This metric therefore matches the real action better than generic accuracy.

The positive proxy rate among eligible pages is approximately 61.64%. A ranking that achieves the same result would not be useful because selecting pages without an effective ranking could produce a similar rate.

My provisional success threshold is `Precision@50 ≥ 0.72`. This means that at least 36 of the top 50 pages must meet the held-out decline proxy. The system must also outperform the transparent fixed-rule baseline using the same metric.

This threshold is provisional and must be frozen before model comparison. Final evaluation should use client-grouped or time-aware held-out data, not the same rows used to construct or tune the score.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
K = 50
REQUIRED_LIFT_OVER_BASE_RATE = 0.10

raw_success_threshold = min(
    1.0,
    proxy_base_rate + REQUIRED_LIFT_OVER_BASE_RATE,
)

required_true_positives = int(
    np.ceil(raw_success_threshold * K)
)

success_precision = required_true_positives / K


def precision_at_k(y_true, scores, k=50):
    """
    Calculate precision among the k highest-scored rows.
    Use only on held-out evaluation data.
    """
    evaluation = pd.DataFrame(
        {
            "y_true": y_true,
            "score": scores,
        }
    ).dropna()

    if len(evaluation) < k:
        raise ValueError(
            f"Need at least {k} evaluated rows; "
            f"received {len(evaluation)}."
        )

    top_k = evaluation.nlargest(k, "score")

    return float(top_k["y_true"].mean())


print(f"Proxy base rate: {proxy_base_rate:.2%}")
print(
    f"Provisional success bar: "
    f"Precision@{K} >= {success_precision:.2%}"
)
print(
    f"That means at least {required_true_positives} "
    f"true proxy positives in the top {K}."
)
print(
    "Final evaluation must use held-out pages, "
    "clients, or later time data."
)

Proxy base rate: 61.64%
Provisional success bar: Precision@50 >= 72.00%
That means at least 36 true proxy positives in the top 50.
Final evaluation must use held-out pages, clients, or later time data.


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one pseudonymized content page at the selected decision point**.

Therefore:

- One row represents one unique `content_id`.
- `content_id` is used only to identify and deduplicate rows.
- `client_id` is a pseudonymized grouping key that can be used for client-grouped validation.
- Neither `content_id` nor `client_id` should be supplied to a model as a predictive feature.
- Previous-period impressions, clicks, and sessions are candidate pre-decision signals.
- Later-period impressions, observed change, and the target proxy are outcome-only columns.
- The outcome-only columns must be excluded from the model feature matrix.

The displayed dataframe is a framing slice used to demonstrate the unit and target. It is not yet the final leakage-audited training dataset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_columns = [
    "content_id",
    "client_id",
    "content_type",
    "main_intent",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "impressions_last_30d",
    "observed_impression_change_pct",
    "future_decline_proxy",
]

lane_df = df.loc[eligible, lane_columns].copy()

assert lane_df["content_id"].is_unique, (
    "The unit-of-analysis check failed: "
    "content_id contains duplicates."
)

candidate_feature_columns = [
    "content_type",
    "main_intent",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

outcome_only_columns = [
    "impressions_last_30d",
    "observed_impression_change_pct",
    "future_decline_proxy",
]

print(
    f"Lane slice shape: "
    f"{lane_df.shape[0]:,} rows x "
    f"{lane_df.shape[1]} columns"
)
print(
    f"Unique content pages: "
    f"{lane_df['content_id'].nunique():,}"
)
print(
    f"Pseudonymized client groups represented: "
    f"{lane_df['client_id'].nunique():,}"
)
print(
    "Unit of analysis: "
    "one row = one pseudonymized content page."
)
print(
    "\nCandidate feature columns:",
    candidate_feature_columns,
)
print(
    "\nOutcome-only columns:",
    outcome_only_columns,
)

lane_df.head(10)

Lane slice shape: 18,010 rows x 10 columns
Unique content pages: 18,010
Pseudonymized client groups represented: 30
Unit of analysis: one row = one pseudonymized content page.

Candidate feature columns: ['content_type', 'main_intent', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

Outcome-only columns: ['impressions_last_30d', 'observed_impression_change_pct', 'future_decline_proxy']


,content_id,client_id,content_type,main_intent,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,impressions_last_30d,observed_impression_change_pct,future_decline_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,987,13,9,578,-41.438703,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,5915,1,2,2501,-57.717667,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,6089,3,3,2382,-60.880276,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,4206,17,26,3626,-13.789824,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,6452,2,9,4211,-34.733416,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,1009,1,1,617,-38.850347,1
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,632,0,4,636,0.632911,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,13828,8,14,5696,-58.808215,1
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,356,0,0,252,-29.213483,1
10,content_d8ee6cc6d642,client_19581e27de,keyword article,commercial,6441,117,136,7665,19.003260,0


## 5. Why ML beats a fixed rule here

A fixed rule remains necessary as the baseline. For example, a simple rule could prioritize pages with enough previous impressions and a large observed decline. Such a rule is transparent and easy to audit.

However, the real ranking problem may involve interactions among demand, clicks, sessions, content type, intent, search position, freshness, content depth, and engagement. The importance of one signal may depend on another. For example, the same impression decline may have different operational importance for a high-demand page and a low-demand page.

A fixed rule requires manually chosen thresholds and usually treats them independently. A learned method may produce a continuous score, capture nonlinear interactions, and rank borderline cases more effectively.

The grouped descriptive check below also shows that observed proxy rates differ across content-type and intent groups. This is directional evidence that one global threshold may not behave equally across the inventory. It is not proof that ML will perform better.

ML earns its place only if it beats the fixed-rule baseline on held-out `Precision@50`, survives leakage checks, and produces reason codes that a reviewer can understand. If it does not, the simpler rule should be retained.

The operational action is human review: an editor inspects the top 50 candidates and uses the reason codes to decide whether to refresh, expand, protect, prune, or monitor each page. The model does not automatically change content.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pattern_table = (
    lane_df
    .groupby(
        ["content_type", "main_intent"],
        dropna=False,
    )
    .agg(
        pages=("content_id", "size"),
        decline_proxy_rate=(
            "future_decline_proxy",
            "mean",
        ),
        median_prior_impressions=(
            "impressions_prev_30d",
            "median",
        ),
        median_prior_clicks=(
            "clicks_prev_30d",
            "median",
        ),
    )
    .reset_index()
)

# Avoid interpreting extremely small groups.
pattern_table = pattern_table.loc[
    pattern_table["pages"].ge(50)
].copy()

pattern_table["decline_proxy_rate_pct"] = (
    pattern_table["decline_proxy_rate"] * 100
).round(2)

pattern_table = (
    pattern_table
    .drop(columns="decline_proxy_rate")
    .sort_values(
        "decline_proxy_rate_pct",
        ascending=False,
    )
)

rate_spread = (
    pattern_table["decline_proxy_rate_pct"].max()
    - pattern_table["decline_proxy_rate_pct"].min()
)

print(
    "Observed proxy-rate spread across groups "
    f"with at least 50 pages: "
    f"{rate_spread:.2f} percentage points."
)
print(
    "This is directional evidence of heterogeneous "
    "patterns, not proof that ML will win."
)

pattern_table

Observed proxy-rate spread across groups with at least 50 pages: 21.98 percentage points.
This is directional evidence of heterogeneous patterns, not proof that ML will win.


,content_type,main_intent,pages,median_prior_impressions,median_prior_clicks,decline_proxy_rate_pct
6,keyword article,NaN,107,216.0,0.0,81.31
0,comparison article,informational,131,163.0,0.0,74.81
1,feedly article,NaN,206,244.0,1.0,68.45
3,keyword article,informational,10804,813.5,1.0,62.26
2,keyword article,commercial,3002,823.5,1.0,60.49
5,keyword article,transactional,3742,864.0,2.0,59.33


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.